In [ ]:
from torchmetrics import MeanSquaredError
from torchmetrics.image import StructuralSimilarityIndexMeasure, MultiScaleStructuralSimilarityIndexMeasure

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torch.optim import Adam

import numpy as np

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64


In [ ]:
from magsr.datasets import build_wa_datasets, pool_collate, worker_init_fn

# Call Western Australia dataset
splits = build_wa_datasets()

def make_loader(split, *, shuffle):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(
        splits[split],
        collate_fn=pool_collate,
        batch_size=BATCH_SIZE,
        num_workers=4,
        worker_init_fn=worker_init_fn,
        shuffle=shuffle,
        generator=g,
        persistent_workers=True,
    )

dl_train, dl_val, dl_test = make_loader("train", shuffle=True), make_loader("val", shuffle=False), make_loader("test", shuffle=False)

vmin, vmax = splits["train"].vmin, splits["train"].vmax
def denormalise(arr):
    """Inverse of the percentile-based normalization in `WA_Dataset`."""
    return arr * (vmax - vmin) + vmin


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange

def peek_batch(loader):
    batch = next(iter(loader))
    b_size = batch["hr"].shape[0]
    
    # Calculate optimal grid dimensions
    n_rows = int(np.ceil(np.sqrt(b_size)))
    n_cols = int(np.ceil(b_size / n_rows))
    n_target = n_rows * n_cols
    
    # Extract, drop C=1, and convert to numpy -> shape: (b_size, H, W)
    hr = batch["hr"].squeeze(1).numpy()
    lr = batch["lr"].squeeze(1).numpy()

    # Pad arrays with zeros if the batch size doesn't perfectly fill the grid
    if b_size < n_target:
        pad_size = n_target - b_size
        hr = np.pad(hr, ((0, pad_size), (0, 0), (0, 0)), mode='constant', constant_values=0)
        lr = np.pad(lr, ((0, pad_size), (0, 0), (0, 0)), mode='constant', constant_values=0)


    hr_grid = rearrange(hr, "(nr nc) h w -> (nr h) (nc w)", nr=n_rows, nc=n_cols)
    lr_grid = rearrange(lr, "(nr nc) h w -> (nr h) (nc w)", nr=n_rows, nc=n_cols)

    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(lr_grid, cmap="gray")
    axes[0].set_title(f"LR mosaic ({n_rows}×{n_cols})")
    
    axes[1].imshow(hr_grid, cmap="gray")
    axes[1].set_title(f"HR mosaic ({n_rows}×{n_cols})")
    
    plt.suptitle(f"WA MAG — first train batch (Size: {b_size})", fontweight="bold")
    plt.tight_layout()
    plt.show()

# Run the function
peek_batch(dl_train)

### Build RDN Model

In [ ]:
from magsr.models import RDNpp, rdnpp_default_x4

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

rdnpp = rdnpp_default_x4().to(DEVICE)
print(f"RDN++ has {count_params(rdnpp):,} trainable parameters")

### Training

In [ ]:
# -----------------------------------------------------------------
# 4.1  Loss functions
# -----------------------------------------------------------------
class MaskedL1Loss(nn.Module):
    """L1 over HR-valid pixels. Mask is derived from NaNs in `target`."""
    def forward(self, pred, target):
        mask = ~torch.isnan(target)
        diff = (pred - target.nan_to_num(0.0)).abs()
        return diff[mask].mean() if mask.any() else diff.sum() * 0.0

class MaskedSSIMLoss(nn.Module):
    def __init__(self, window_size=11, sigma=1.5):
        super().__init__()
        k = torch.arange(window_size, dtype=torch.float32) - window_size//2
        g = torch.exp(-k**2 / (2*sigma**2)); g /= g.sum()
        self.register_buffer('kernel', g.outer(g)[None, None])
        self.pad = window_size // 2

    def forward(self, pred, target, mask):
        pred = pred * mask; target = target * mask
        C1, C2 = 0.01**2, 0.03**2; p = self.pad
        mu_x  = F.conv2d(pred,          self.kernel, padding=p)
        mu_y  = F.conv2d(target,         self.kernel, padding=p)
        sig_x  = F.conv2d(pred*pred,     self.kernel, padding=p) - mu_x*mu_x
        sig_y  = F.conv2d(target*target, self.kernel, padding=p) - mu_y*mu_y
        sig_xy = F.conv2d(pred*target,   self.kernel, padding=p) - mu_x*mu_y
        ssim   = ((2*mu_x*mu_y+C1)*(2*sig_xy+C2)) / ((mu_x**2+mu_y**2+C1)*(sig_x+sig_y+C2))
        return (1 - ssim).mean()
    
criterion = MaskedL1Loss().to(DEVICE)

In [ ]:
import math
from torch.optim import Adam
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm
from pathlib import Path

import wandb
from torchmetrics import MeanSquaredError
from torchmetrics.image import (
    StructuralSimilarityIndexMeasure,
    MultiScaleStructuralSimilarityIndexMeasure,
)

# --- Configuration from Smith et al. (2022) ---
NUM_EPOCHS = 602
GRAD_CLIP = 1.0
LR_START = 3e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path('checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

WANDB_PROJECT = 'magsr-wa'
WANDB_ENTITY = None  # set to your wandb entity/team or leave None for default

wandb.login()  # no-op if already logged in; prompts for API key otherwise

optim = Adam(rdnpp.parameters(), lr=LR_START)

sched = OneCycleLR(
    optim,
    max_lr=LR_START,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(dl_train),
    pct_start=0.3,
    anneal_strategy='cos',
)


def train_model(model, name, train_dl, val_dl, criterion):
    """
    Args:
        model: RDN model.
        name: Run name (used for wandb + checkpoint filenames).
        train_dl: Training dataloader.
        val_dl: Validation dataloader.
        criterion: Loss callable taking (pred, target).
    """
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=name,
        config={
            'model': type(model).__name__,
            'num_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
            'epochs': NUM_EPOCHS,
            'batch_size': BATCH_SIZE,
            'lr_start': LR_START,
            'grad_clip': GRAD_CLIP,
            'optimizer': 'Adam',
            'scheduler': 'OneCycleLR',
            'pct_start': 0.3,
            'anneal_strategy': 'cos',
            'criterion': type(criterion).__name__,
            'seed': SEED,
        },
    )
    wandb.watch(model, log='gradients', log_freq=200)

    val_mse   = MeanSquaredError().to(DEVICE)
    val_ssim  = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)
    val_msssim = MultiScaleStructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)

    hist = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': [], 'val_msssim': []}
    best_ssim = -float('inf')
    global_step = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        t_loss = 0.0
        pbar = tqdm(train_dl, desc=f'[{name}] {epoch:03d}/{NUM_EPOCHS}', leave=False, ncols=100)

        for batch in pbar:
            lr_t  = batch['lr'].to(DEVICE, non_blocking=True)
            hr_t  = batch['hr'].to(DEVICE, non_blocking=True)

            optim.zero_grad(set_to_none=True)

            pred = model(lr_t)
            loss = criterion(pred, hr_t)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optim.step()
            sched.step()

            t_loss += loss.item()
            current_lr = sched.get_last_lr()[0]
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.2e}'})

            wandb.log(
                {'train/loss_step': loss.item(), 'train/lr': current_lr, 'epoch': epoch},
                step=global_step,
            )
            global_step += 1

        t_loss /= len(train_dl)

        # --- validate ---
        model.eval()
        v_loss = 0.0
        val_mse.reset(); val_ssim.reset(); val_msssim.reset()

        with torch.no_grad():
            for batch in val_dl:
                lr_t  = batch['lr'].to(DEVICE, non_blocking=True)
                hr_t  = batch['hr'].to(DEVICE, non_blocking=True)
                msk_t = hr_t.isfinite().float()  # mask derived from NaNs in HR

                sr = model(lr_t).clamp(0, 1)
                v_loss += criterion(sr, hr_t).item()

                # zero invalid pixels in both tensors so masked regions don't distort metrics
                sr_m = sr * msk_t
                hr_m = hr_t * msk_t
                val_mse.update(sr_m, hr_m)
                val_ssim.update(sr_m, hr_m)
                val_msssim.update(sr_m, hr_m)

        v_loss /= len(val_dl)
        v_mse_val = val_mse.compute().item()
        v_psnr = 10.0 * math.log10(1.0 / (v_mse_val + 1e-12))
        v_ssim = val_ssim.compute().item()
        v_msssim = val_msssim.compute().item()

        hist['train_loss'].append(t_loss)
        hist['val_loss'].append(v_loss)
        hist['val_psnr'].append(v_psnr)
        hist['val_ssim'].append(v_ssim)
        hist['val_msssim'].append(v_msssim)

        wandb.log(
            {
                'epoch': epoch,
                'train/loss_epoch': t_loss,
                'val/loss': v_loss,
                'val/mse': v_mse_val,
                'val/psnr': v_psnr,
                'val/ssim': v_ssim,
                'val/msssim': v_msssim,
            },
            step=global_step,
        )

        # --- checkpoints ---
        ckpt = {
            'epoch': epoch,
            'model': model.state_dict(),
            'optim': optim.state_dict(),
            'sched': sched.state_dict(),
            'hist': hist,
            'val_psnr': v_psnr,
            'val_ssim': v_ssim,
            'val_msssim': v_msssim,
        }
        torch.save(ckpt, CKPT_DIR / f'{name}_last.pt')
        if v_ssim > best_ssim:
            best_ssim = v_ssim
            torch.save(ckpt, CKPT_DIR / f'{name}_best.pt')
            wandb.run.summary['best/val_ssim'] = v_ssim
            wandb.run.summary['best/val_psnr'] = v_psnr
            wandb.run.summary['best/val_msssim'] = v_msssim
            wandb.run.summary['best/epoch'] = epoch

    run.finish()
    return hist


In [ ]:
# Train the model
hist = train_model(rdnpp, 'rdnpp_x4', dl_train, dl_val, criterion)